# Network Intrusion Detection: Data Understanding and EDA

This notebook loads the NSL-KDD train and test files, validates label mappings, and explores attack distributions for binary and multiclass intrusion detection.


In [1]:
import json
import sys
from pathlib import Path

PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

print(PROJECT_ROOT)


/Users/paco/Documents/Git Projects/network-intrusion-detection-system


In [2]:
import pandas as pd

from src.config import TRAIN_FILE, TEST_FILE, TEST_21_FILE, TRAIN_20_FILE, FIGURES_DIR, ensure_project_dirs
from src.data import load_nsl_kdd, summarize_labels, validate_raw_files
from src.visualize import save_countplot

ensure_project_dirs()
validate_raw_files([TRAIN_FILE, TEST_FILE, TRAIN_20_FILE, TEST_21_FILE])


In [3]:
train_df = load_nsl_kdd(TRAIN_FILE)
test_df = load_nsl_kdd(TEST_FILE)
test_21_df = load_nsl_kdd(TEST_21_FILE)

print("Train:", train_df.shape)
print("Test:", test_df.shape)
print("Hard test:", test_21_df.shape)
display(train_df.head())


Train: (125973, 45)
Test: (22544, 45)
Hard test: (11850, 45)


,duration,protocol_type,service,flag,src_bytes,dst_bytes,land,wrong_fragment,urgent,hot,...,dst_host_same_src_port_rate,dst_host_srv_diff_host_rate,dst_host_serror_rate,dst_host_srv_serror_rate,dst_host_rerror_rate,dst_host_srv_rerror_rate,attack_type,difficulty,binary_label,attack_category
0,0,tcp,ftp_data,SF,491,0,0,0,0,0,...,0.17,0.00,0.00,0.00,0.05,0.00,normal,20,0,normal
1,0,udp,other,SF,146,0,0,0,0,0,...,0.88,0.00,0.00,0.00,0.00,0.00,normal,15,0,normal
2,0,tcp,private,S0,0,0,0,0,0,0,...,0.00,0.00,1.00,1.00,0.00,0.00,neptune,19,1,dos
3,0,tcp,http,SF,232,8153,0,0,0,0,...,0.03,0.04,0.03,0.01,0.00,0.01,normal,21,0,normal
4,0,tcp,http,SF,199,420,0,0,0,0,...,0.00,0.00,0.00,0.00,0.00,0.00,normal,21,0,normal


## Binary Class Distribution


In [4]:
binary_train = summarize_labels(train_df, "binary_label")
binary_test = summarize_labels(test_df, "binary_label")

display(binary_train)
display(binary_test)

save_countplot(train_df, "binary_label", FIGURES_DIR / "binary_distribution_train.png", "Binary Label Distribution - Train")
save_countplot(test_df, "binary_label", FIGURES_DIR / "binary_distribution_test.png", "Binary Label Distribution - Test")


,binary_label,count,percentage
0,0,67343,53.458281
1,1,58630,46.541719


,binary_label,count,percentage
0,1,12833,56.924237
1,0,9711,43.075763


## Attack Category Distribution


In [5]:
category_train = summarize_labels(train_df, "attack_category")
category_test = summarize_labels(test_df, "attack_category")

display(category_train)
display(category_test)

save_countplot(train_df, "attack_category", FIGURES_DIR / "attack_category_train.png", "Attack Categories - Train")
save_countplot(test_df, "attack_category", FIGURES_DIR / "attack_category_test.png", "Attack Categories - Test")


,attack_category,count,percentage
0,normal,67343,53.458281
1,dos,45927,36.457812
2,probe,11656,9.252776
3,r2l,995,0.789852
4,u2r,52,0.041279


,attack_category,count,percentage
0,normal,9711,43.075763
1,dos,7460,33.090845
2,r2l,2885,12.797197
3,probe,2421,10.738999
4,u2r,67,0.297197


## Attack Type Coverage


In [6]:
train_attacks = set(train_df["attack_type"].unique())
test_attacks = set(test_df["attack_type"].unique())

print("Train attack types:", len(train_attacks))
print("Test attack types:", len(test_attacks))
print("Attack types only in test:")
print(sorted(test_attacks - train_attacks))


Train attack types: 23
Test attack types: 38
Attack types only in test:
['apache2', 'httptunnel', 'mailbomb', 'mscan', 'named', 'processtable', 'ps', 'saint', 'sendmail', 'snmpgetattack', 'snmpguess', 'sqlattack', 'udpstorm', 'worm', 'xlock', 'xsnoop', 'xterm']


## Key EDA Notes


In [7]:
eda_summary = {
    "train_rows": len(train_df),
    "test_rows": len(test_df),
    "hard_test_rows": len(test_21_df),
    "train_attack_rate": float(train_df["binary_label"].mean()),
    "test_attack_rate": float(test_df["binary_label"].mean()),
    "train_attack_types": len(train_attacks),
    "test_attack_types": len(test_attacks),
    "test_only_attack_types": sorted(test_attacks - train_attacks),
}

with open(PROJECT_ROOT / "reports" / "eda_summary.json", "w", encoding="utf-8") as file:
    json.dump(eda_summary, file, indent=2)

eda_summary


{'train_rows': 125973,
 'test_rows': 22544,
 'hard_test_rows': 11850,
 'train_attack_rate': 0.4654171925730117,
 'test_attack_rate': 0.5692423704755145,
 'train_attack_types': 23,
 'test_attack_types': 38,
 'test_only_attack_types': ['apache2',
  'httptunnel',
  'mailbomb',
  'mscan',
  'named',
  'processtable',
  'ps',
  'saint',
  'sendmail',
  'snmpgetattack',
  'snmpguess',
  'sqlattack',
  'udpstorm',
  'worm',
  'xlock',
  'xsnoop',
  'xterm']}